In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("retail_store_sales.csv")

In [5]:
df_clean = df.copy()

In [6]:
df_clean.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [7]:
df_clean.shape[0]

12575

In [9]:
df_clean.drop_duplicates(inplace=True)

In [10]:
df_clean.shape[0]

12575

In [11]:
df_clean.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


In [13]:
df_clean.dropna(how = 'all', inplace=True) # remove completely empty rows

In [17]:
for col in df_clean.select_dtypes(include = ["float64", 'int64']).columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

/tmp/ipykernel_6258/2452223494.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(df_clean[col].median(), inplace=True)


In [18]:
for col in df_clean.select_dtypes(include = 'object').columns:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

/tmp/ipykernel_6258/687606161.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
/tmp/ipykernel_6258/687606161.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)


In [19]:
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'])

In [20]:
# Clean column names
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ", "_")

df_clean.columns

Index(['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit',
       'quantity', 'total_spent', 'payment_method', 'location',
       'transaction_date', 'discount_applied'],
      dtype='object')

In [21]:
df_clean.isnull().sum()

,0
transaction_id,0
customer_id,0
category,0
item,0
price_per_unit,0
quantity,0
total_spent,0
payment_method,0
location,0
transaction_date,0


In [22]:
# Remove extreme outliers using IQR
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns

In [24]:
for col in numeric_cols:
  Q1 = df_clean[col].quantile(0.25)
  Q2 = df_clean[col].quantile(0.50)
  Q3 = df_clean[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]


In [25]:
print("Final Shape:", df_clean.shape)

Final Shape: (12418, 11)


In [27]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12418 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    12418 non-null  object        
 1   customer_id       12418 non-null  object        
 2   category          12418 non-null  object        
 3   item              12418 non-null  object        
 4   price_per_unit    12418 non-null  float64       
 5   quantity          12418 non-null  float64       
 6   total_spent       12418 non-null  float64       
 7   payment_method    12418 non-null  object        
 8   location          12418 non-null  object        
 9   transaction_date  12418 non-null  datetime64[ns]
 10  discount_applied  12418 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(3), object(6)
memory usage: 1.1+ MB


In [28]:
df_clean.describe()

,price_per_unit,quantity,total_spent,transaction_date
count,12418.000000,12418.000000,12418.000000,12418
mean,23.155701,5.502496,125.266549,2023-07-13 01:45:10.581414144
min,5.000000,1.000000,5.000000,2022-01-01 00:00:00
25%,14.000000,3.000000,55.000000,2022-09-30 00:00:00
50%,23.000000,6.000000,108.500000,2023-07-13 00:00:00
75%,32.000000,8.000000,182.000000,2024-04-24 00:00:00
max,41.000000,10.000000,369.000000,2025-01-18 00:00:00
std,10.394878,2.762402,88.109931,NaN


In [30]:
import os

# Create folder (important in Colab)
os.makedirs("data/processed", exist_ok=True)

# Save cleaned data
df_clean.to_csv("data/processed/cleaned_data.csv", index=False)

print("File saved successfully!")

File saved successfully!


In [31]:
from google.colab import files

files.download("data/processed/cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>